# 01 — In Silico Digestion Exploration

This notebook tests the first end-to-end step of the HERALD computational pipeline: retrieving whey protein sequences and simulating enzymatic digestion.

The goal is to confirm that the current code can:

1. Retrieve target whey protein sequences from UniProt.
2. Apply enzyme-specific cleavage rules.
3. Generate predicted peptide fragments from each protein sequence.
4. Compare peptide profiles produced by different enzymes.

At this stage, the workflow does **not** yet predict antimicrobial activity. It only generates the peptide fragments that will later be scored for AMP-like properties.

Import HERALD Functions

This section imports the core functions developed so far.

* protein_sequence() retrieves a protein sequence from UniProt or the local cache.
* WHEY_PROTEINS stores the target whey protein names and their UniProt accession IDs.
* digest_sequence() simulates enzymatic digestion using the cleavage rules defined in enzymes.py.

In [ ]:
import sys
import os

PROJECT_ROOT = os.path.abspath("..")
sys.path.append(PROJECT_ROOT)

from herald.proteins import protein_sequence, WHEY_PROTEINS
from herald.digestion import digest_sequence

Retrieve Whey Protein Sequences

Here, we retrieve the amino-acid sequences for the target whey proteins.

Each protein is identified by a UniProt accession ID. The sequence is either fetched directly from UniProt or loaded from the local cache if it has already been downloaded.

This step verifies that the protein-fetching module is working correctly and that the target proteins are available for downstream digestion analysis.

In [4]:
for protein_name, accession_id in WHEY_PROTEINS.items():
    sequence = protein_sequence(accession_id)
    print(protein_name, accession_id, len(sequence))

beta-lactoglobulin P02754 178
alpha-lactalbumin P00709 142
lactoferrin P24627 708
BSA P02769 607


## Simulate Trypsin Digestion of β-Lactoglobulin

In this section, we simulate digestion of β-lactoglobulin using trypsin.

Trypsin is a sequence-specific protease that primarily cleaves after lysine (`K`) and arginine (`R`), unless the next amino acid is proline (`P`). The resulting peptide fragments represent the predicted products of an in silico hydrolysis reaction.

For this initial test, only peptides between 4 and 50 amino acids are retained. This range removes very short fragments while keeping peptide-sized products that may later be evaluated for AMP-like properties.

In [5]:
beta_lg = protein_sequence(WHEY_PROTEINS["beta-lactoglobulin"])

trypsin_peptides = digest_sequence(
    beta_lg,
    "trypsin",
    min_length=4,
    max_length=50,
)

print(f"Number of trypsin peptides: {len(trypsin_peptides)}")
print("Trypsin peptides:", trypsin_peptides[:10])

Number of trypsin peptides: 13


['CLLLALALTCGAQALIVTQTMK',
 'GLDIQK',
 'VAGTWYSLAMAASDISLLDAQSAPLR',
 'VYVEELKPTPEGDLEILLQK',
 'WENGECAQK',
 'IIAEK',
 'IPAVFK',
 'IDALNENK',
 'VLVLDTDYK',
 'YLLFCMENSAEPEQSLACQCLVR']

## Compare Trypsin and Chymotrypsin Digestion

Different enzymes cleave proteins at different amino-acid positions. As a result, the same starting protein can produce different peptide profiles depending on the enzyme used.

Here, we compare predicted peptide fragments from β-lactoglobulin digested with:

- **Trypsin**, which cleaves primarily after `K` and `R`
- **Chymotrypsin**, which cleaves primarily after aromatic or hydrophobic residues such as `F`, `Y`, `W`, and `L`

This comparison provides an initial check that the enzyme-specific cleavage rules produce different computational digestion outcomes.

In [8]:
chymotrypsin_peptides = digest_sequence(
    beta_lg,
    "chymotrypsin",
    min_length=4,
    max_length=50,
)

print(f"Number of trypsin peptides: {len(trypsin_peptides)}")
print(f"Number of chymotrypsin peptides: {len(chymotrypsin_peptides)}")
print("Chymotrypsin peptides:", chymotrypsin_peptides[:10])

Number of trypsin peptides: 13
Number of chymotrypsin peptides: 19
Chymotrypsin peptides: ['MKCL', 'TCGAQAL', 'IVTQTMKGL', 'DIQKVAGTW', 'AMAASDISL', 'DAQSAPL', 'VEEL', 'KPTPEGDL', 'ENGECAQKKIIAEKTKIPAVF', 'KIDAL']
